In [38]:
import pandas as pd
import numpy as np

from sklearn import preprocessing
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

from xailib.data_loaders.dataframe_loader import prepare_dataframe

from xailib.explainers.lime_explainer import LimeXAITabularExplainer
from xailib.explainers.lore_explainer import LoreTabularExplainer
from xailib.explainers.shap_explainer_tab import ShapXAITabularExplainer

from xailib.models.sklearn_classifier_wrapper import sklearn_classifier_wrapper

import altair as alt

import os

In [39]:
from plot_explanation import PlotExplanation
import pickle

In [40]:
path = os.getcwd()
print(path)

/home/sax/PycharmProjects/xai-visualization_rules_fi/notebooks


# Loading a dataset and preparing it

In [41]:
datasets=['titanic_c.csv','german_credit.csv']

In [42]:
source_file = f'../datasets/{datasets[1]}'
class_field = 'default'
# Load and transform dataset
df = pd.read_csv(source_file, skipinitialspace=True, na_values='?', keep_default_na=True)

In [43]:
df, feature_names, class_values, numeric_columns, rdf, real_feature_names, features_map = prepare_dataframe(df, class_field)

## Learning a Random Forest classfier

We train a RF classifier by using the ```sklearn``` library. We start by splitting the dataset into a train and test subsets. 

In [44]:
test_size = 0.3
random_state = 42
X_train, X_test, Y_train, Y_test = train_test_split(df[feature_names], df[class_field],
                                                        test_size=test_size,
                                                        random_state=random_state,
                                                        stratify=df[class_field])


Then we train the model on the training set. 
Once the model has been learned, we use a wrapper class to get access to the model for ```XAI lib```

In [45]:
bb = RandomForestClassifier(n_estimators=20, random_state=random_state)
bb.fit(X_train.values, Y_train.values)
bbox = sklearn_classifier_wrapper(bb)

Select a new instance to be classfied by the model and print the predicted class.

In [46]:
inst = X_train.iloc[182].values
print('Instance ',inst)
print('True class ',Y_train.iloc[128])
print('Predicted class ',bb.predict(inst.reshape(1, -1)))

Instance  [   48 12169     4     4    36     1     1     1     0     0     0     1
     0     0     0     0     0     0     1     0     0     0     0     0
     0     0     0     0     0     0     1     0     0     0     0     1
     0     0     0     1     1     0     0     0     0     0     1     0
     1     0     1     0     0     1     0     0     0     0     1     0
     1]
True class  0
Predicted class  [0]


In [47]:
real_inst = inst
real_inst

array([   48, 12169,     4,     4,    36,     1,     1,     1,     0,
           0,     0,     1,     0,     0,     0,     0,     0,     0,
           1,     0,     0,     0,     0,     0,     0,     0,     0,
           0,     0,     0,     1,     0,     0,     0,     0,     1,
           0,     0,     0,     1,     1,     0,     0,     0,     0,
           0,     1,     0,     1,     0,     1,     0,     0,     1,
           0,     0,     0,     0,     1,     0,     1])

## Explaining the prediction
We use the explanators of ```XAI lib``` to provide an explantion for the classified instance ```inst```.
Every explainer of ```XAI lib``` takes in input the blackbox to be explained with the corresponding feature names, and a configuration object to initialize the explainer.

### SHAP explainer

In [48]:
explainer = ShapXAITabularExplainer(bbox, feature_names)
config = {'explainer' : 'tree', 'X_train' : X_train.iloc[0:].values}
explainer.fit(config)

In [49]:
exp = explainer.explain(inst)

In [50]:
shap_feature_importance=exp.exp

## Learning a different model

### Learning a Logistic Regressor

We train a Logistic Regression by using the ```sklearn``` library. We transform the dataset by using a ```Scaler``` to normalize all the attributes.

In [51]:
scaler = preprocessing.StandardScaler().fit(X_train)
X_scaled = scaler.transform(X_train)

bb = LogisticRegression(C=1, penalty='l2')
bb.fit(X_scaled, Y_train.values)
# pass the model to the wrapper to use it in the XAI lib
bbox = sklearn_classifier_wrapper(bb)

In [52]:
# select a record to explain
inst_s = X_scaled[182]
print('Instance ',inst_s)
print('Predicted class ',bb.predict(inst.reshape(1, -1)))

Instance  [ 2.27797454  3.35504085  0.94540357  1.07634233  0.04854891 -0.72456474
 -0.43411405  1.65027399 -0.61477862 -0.25898489 -0.80681063  4.17385345
 -0.6435382  -0.32533856 -1.03489416 -0.20412415 -0.22941573 -0.33068147
  1.75885396 -0.34899122 -0.60155441 -0.15294382 -0.09298136 -0.46852129
 -0.12038585 -0.08481889 -0.23623492 -1.21387736 -0.36174054 -0.24943031
  2.15526362 -0.59715086 -0.45485883 -0.73610476 -0.43875307  4.23307441
 -0.65242771 -0.23958675 -0.32533856  0.90192655  4.72581563 -0.2259448
 -3.15238005 -0.54212562 -0.70181003 -0.63024248  2.30354212 -0.40586384
  0.49329429 -0.23958675  2.88675135 -1.59227935 -0.46170508  2.46388049
 -1.33747696 -0.13206764 -0.5        -1.21387736  1.21387736 -0.20412415
  0.20412415]
Predicted class  [1]


## Explaining the prediction
We use the same explainators as for the previous model. In this case, a few adjustments are necessary for the initialization of the explanators. For example, SHAP needs a specific configuration for the linear model we are using.

## LIME tabular explainer

In [53]:
limeExplainer = LimeXAITabularExplainer(bbox)
config = {'feature_selection': 'lasso_path'}
limeExplainer.fit(df, class_field, config)
lime_exp = limeExplainer.explain(inst)

In [54]:
lime_feature_importance=lime_exp.exp.as_list()
lime_feature_importance[1:3]

[('other_debtors=co-applicant', -0.023447991894281556),
 ('personal_status_sex=male : married/widowed', 0.018628269422834885)]

In [55]:
#lime_exp.plot_features_importance()

### LORE explainer

In [56]:
explainer = LoreTabularExplainer(bbox)
config = {'neigh_type':'geneticp', 'size':1000, 'ocr':0.1, 'ngen':10}
explainer.fit(df, class_field, config)

exp = explainer.explain(inst)
print(exp)

In [57]:
expDict = exp.expDict
# remove key dt from expDict
expDict.pop('dt', None)
expDict

{'bb_pred': 1,
 'dt_pred': 1,
 'rule': {'premise': [{'att': 'credit_amount',
    'op': '>',
    'thr': 8.5,
    'is_continuous': True}],
  'cons': 1,
  'class_name': 'default'},
 'crules': [],
 'deltas': [],
 'fidelity': 0.9991348371174513}

In [58]:
len(inst)

61

In [59]:
len(feature_names)

61

In [60]:
inst

array([   48, 12169,     4,     4,    36,     1,     1,     1,     0,
           0,     0,     1,     0,     0,     0,     0,     0,     0,
           1,     0,     0,     0,     0,     0,     0,     0,     0,
           0,     0,     0,     1,     0,     0,     0,     0,     1,
           0,     0,     0,     1,     1,     0,     0,     0,     0,
           0,     1,     0,     1,     0,     1,     0,     0,     1,
           0,     0,     0,     0,     1,     0,     1])

In [61]:
print(exp)

In [62]:
exp_1=explainer.explain(inst)
print(exp_1)

In [63]:
rules =exp.expDict['rule']['premise']

In [64]:
rules[0]

{'att': 'credit_amount', 'op': '>', 'thr': 8.5, 'is_continuous': True}

# Plotting functions

In [65]:
def single_feature_importance_plot(dataframe, rw):
    data = dataframe[dataframe['name'] == rw['name']]
    chart = alt.Chart(
        data
    ).mark_bar(
    ).encode(
        x=alt.X(
            field='feature_importance',
            type='quantitative',
            title=None
        ),
        y=alt.Y(
            field='name',
            type='nominal',
            title=None,
            axis=None
        ),
        color=alt.condition('datum.feature_importance > 0', alt.value('#285588'), alt.value('#E36273')),
        tooltip=[alt.Tooltip(field="name"),alt.Tooltip(field="feature_importance")]
    )
    return chart.properties(
        height=20,
        width=100
    )

In [66]:
def single_rule_plot_numeric(dataframe,rw):
    data = dataframe[dataframe['name'] == rw['name']]
    p=alt.Chart(
        data
    ).mark_point(
        color='black' if rw['is_continuous'] == True else 'black',
        size=70,
        shape='diamond',
        filled=True
    ).encode(
        x=alt.X(
            field='inst',
            type='quantitative',
            title=None,
            scale= alt.Scale(domain=(rw['min'], rw['max']), clamp=True, nice=False)
        ),
        tooltip=[alt.Tooltip(field='inst', title=rw['name'])]
    )

    t_min = alt.Chart(
        data
    ).mark_text(
        color='black',
        dx=-10,
        align='right'
    ).encode(
        x=alt.X(
            field='min',
            type='quantitative',
            title=None
        ),
        text='min:N'
    )

    t_max = alt.Chart(
        data
    ).mark_text(
        color='black',
        dx=10,
        align='left'
    ).encode(
        x=alt.X(
            field='max',
            type='quantitative',
            title=None
        ),
        text='max:N'
    )

    

    b =alt.Chart(
        data
    ).mark_bar(
        color='#fcc40f',size=5,
        stroke='white'
    ).encode(
        x=alt.X(
            field='thr',
            type='quantitative',
            title=None,
        ),
        x2='thr2',
        y=alt.Y(
            field='name',
            type='nominal',
            title=None
        ),

    )



    l =alt.Chart(
        data
    ).mark_bar(
        color='grey',size=1
    ).encode(
        x=alt.X(
            field='min',
            type='quantitative',
            title=None,
            scale= alt.Scale(domain=(rw['min'], rw['max']), clamp=True, nice=False)
        ),
        x2='max',
        y=alt.Y(field='name',type='nominal',title=None, axis=alt.Axis(labels= False, ticks=False))
    )


    q1_m = alt.Chart(
        data
    ).mark_bar(
        stroke='white',
        color='lightgrey',
        size=18
    ).encode(
        x=alt.X(
            field='q1',
            type='quantitative',
            title=None,
            scale= alt.Scale(domain=(rw['min'], rw['max']), clamp=True, nice=False)
        ),
        x2 = alt.X2(
            field='median'
        ),
    )

    m_q3 = alt.Chart(
        data
    ).mark_bar(
        stroke='white',
        color='lightgrey',
        size=18,
    ).encode(
        x=alt.X(
            field='median',
            type='quantitative',
            title=None,
            scale= alt.Scale(domain=(rw['min'], rw['max']), clamp=True, nice=False)
        ),
        x2 = alt.X2(
            field='q3'
        )
    )
    
    return alt.layer(l,q1_m,m_q3,b,p).properties(
        height=20,
        width=300        
    )

In [67]:
def single_index_text(dataframe, rw):
    data = dataframe[dataframe['name'] == rw['name']]
    chart = alt.Chart(
        data
    ).transform_calculate(
        label ="datum.type=='categorical' ? datum.name : datum.name +' = '+ datum.inst" #  datum.index +' = '+ datum.inst
    ).mark_text(
        color='black',
        align='left',
        dx=-50,
        fontSize=13
    ).encode(
            text=alt.Text(
            field='label',
            type='nominal',
            title=None
        )
    )
    return chart.properties(
        height=20,
        width=101
    )

In [68]:
def plot_rules(dataframe, only_rules=False):
    ti_list=[]
    rp_list=[]
    fi_list=[]
    
    for i, row in dataframe.iterrows():
        if row['inst']!=0:
            if ((only_rules == True) and (row['is_continuous']!= True)):
                pass
            else:
                sti = single_index_text(dataframe, row)
                if row['type']== 'numeric':
                    srp = single_rule_plot_numeric(dataframe, row)
                else:
                    srp = single_rule_plot_qualit(dataframe, row)
                sfi = single_feature_importance_plot(dataframe, row)
                ti_list.append(sti)
                rp_list.append(srp)
                fi_list.append(sfi)
    ti_concat=alt.vconcat(*ti_list)
    rp_concat=alt.vconcat(*rp_list, title='Rule')
    fi_concat=alt.vconcat(*fi_list, title='FI').resolve_scale(
    x='shared'
)
    final_chart = alt.hconcat(
        fi_concat, rp_concat, ti_concat
    )

    return final_chart.configure(
       # background='#F5F5F5',
        padding=20
    ).configure_concat(
        spacing=3
    ).configure_axis(
        grid=False
    ).configure_view(
        strokeWidth=0,
        stroke='lightgray'
    ).configure_axisX(
        disable=True
    ).configure_axisY(
        domain=False,
        ticks=False
    ).configure_title(
        fontWeight='bold',
        anchor='middle',

    )

In [69]:
def single_rule_plot_qualit(dataframe, rw):
    name= rw['name'].split('=')[0]
    data = dataframe[dataframe['rname'] == name]
    
    base= alt.Chart(
        data
    ).transform_stack(
        stack='count',
        as_=['count_start','count_end'],
        groupby=['rname'],
        sort=[alt.SortField('count', 'descending')]
    ).transform_calculate(
        midStack='(datum.count_start+datum.count_end)/2'
    )
    
    
    bar = base.mark_bar(
        stroke='white',
        color='lightgrey'
    ).encode(
        x='count_start:Q',
        x2='count_end:Q',
        tooltip=[alt.Tooltip(field='category', title=name), alt.Tooltip(field='count')]
    )
    
    bar_r = base.mark_bar(
         stroke='#fcc40f'
     ).encode(
        x='count_start:Q',
        x2='count_end:Q',
        color=alt.condition('datum.is_continuous && datum.inst==1',alt.value("#fcc40f"),alt.value('white')),
        opacity=alt.condition('datum.is_continuous',alt.value(1),alt.value(0.0001)),
        tooltip=[alt.Tooltip(field='category', title=name), alt.Tooltip(field='count')]
    )

    r=base.mark_bar(
        stroke='white'
    ).encode(
        x=alt.X(
            field='count',
            type='quantitative',
            title=None,
        ),
        y=alt.Y(
            field='rname',
            type='nominal',
            axis=None
        ),
        detail='name:N',
        color=alt.condition('datum.is_continuous',alt.value('#fcc40f'),alt.value('white')),
        opacity=alt.condition('datum.is_continuous',alt.value(1),alt.value(0.001)),
        tooltip=[alt.Tooltip(field='category', title=name), alt.Tooltip(field='count')]
    )
    
    dot =base.mark_point(
        size=70,
        shape='diamond',
        color='black',
        filled=True
    ).encode(
        x=alt.X(
            field='midStack',
            type='quantitative',
            title=None,
        ),
        y=alt.Y(
            field='rname',
            type='nominal',
            axis=None
        ),
        opacity=alt.condition('datum.inst==1',alt.value(0.6),alt.value(0))
    )
    

    return alt.layer(bar,bar_r,dot).properties(
        height=20,
        width=300,
    )

## Prepare data to plot

In [70]:
pe = PlotExplanation(
    feature_names=feature_names,
    real_feature_names=real_feature_names,
    instance_number=3,
    x_train=X_train,
    rules=rules, 
    feature_importance_type='shap',
    feature_importance=shap_feature_importance,
    numeric_columns=numeric_columns
)

In [71]:
df_v = pe.prepare_dataframe()

In [72]:
df_v

,type,name,rname,min,max,q1,median,q3,mean,std,feature_importance,category,count,inst,op,thr,is_continuous,thr2
30,categorical,savings=unknown/ no savings account,savings,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.079534,unknown/ no savings account,124.0,0,NaN,NaN,NaN,NaN
27,categorical,savings=... < 100 DM,savings,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.073424,... < 100 DM,417.0,1,NaN,NaN,NaN,NaN
7,categorical,account_check_status=0 <= ... < 200 DM,account_check_status,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.038727,0 <= ... < 200 DM,188.0,1,NaN,NaN,NaN,NaN
10,categorical,account_check_status=no checking account,account_check_status,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.030001,no checking account,276.0,0,NaN,NaN,NaN,NaN
4,numeric,age,age,19.0,75.0,27.0,33.0,41.25,35.468571,10.954080,-0.028654,NaN,NaN,20,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
17,categorical,purpose=business,purpose,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.000193,business,69.0,0,NaN,NaN,NaN,NaN
55,categorical,job=unemployed/ unskilled - non-resident,job,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.000167,unemployed/ unskilled - non-resident,12.0,0,NaN,NaN,NaN,NaN
9,categorical,account_check_status=>= 200 DM / salary assign...,account_check_status,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.000107,>= 200 DM / salary assignments for at least 1 ...,44.0,0,NaN,NaN,NaN,NaN
22,categorical,purpose=furniture/equipment,purpose,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000071,furniture/equipment,6.0,0,NaN,NaN,NaN,NaN


In [73]:
pe.prepare_rule_descriptor()

{'duration_in_month': {'eda': {'type': 'numeric',
   'name': 'duration_in_month',
   'rname': 'duration_in_month',
   'min': 4,
   'max': 60,
   'q1': 12.0,
   'median': 18.0,
   'q3': 24.0,
   'mean': 20.768571428571427,
   'std': 11.962776588540619,
   'feature_importance': 0.01802464618992257},
  'rule': [],
  'crules': {},
  'instance_value': 24},
 'credit_amount': {'eda': {'type': 'numeric',
   'name': 'credit_amount',
   'rname': 'credit_amount',
   'min': 338,
   'max': 15945,
   'q1': 1360.75,
   'median': 2319.5,
   'q3': 3974.5,
   'mean': 3200.8728571428574,
   'std': 2674.9420417248216,
   'feature_importance': 0.015363427554491521},
  'rule': [{'att': 'credit_amount',
    'op': '>',
    'thr': 8.5,
    'is_continuous': True}],
  'crules': {},
  'instance_value': 1967},
 'installment_as_income_perc': {'eda': {'type': 'numeric',
   'name': 'installment_as_income_perc',
   'rname': 'installment_as_income_perc',
   'min': 1,
   'max': 4,
   'q1': 2.0,
   'median': 3.0,
   'q3'

# Plot

In [74]:
rules_all=plot_rules(df_v, only_rules=False)
rules_all

alt.HConcatChart(...)

In [29]:
rules_only = plot_rules(df_v, only_rules=True)
rules_only

alt.HConcatChart(...)